# 04 - Model Registry & SPCS Inference Deployment

This notebook covers:
1. Registering the trained model to Snowflake Model Registry
2. Deploying a real-time inference service (SPCS) with REST gateway
3. Testing predictions via SQL, Python, and REST API

**Key Concepts:**
- **Model Registry**: Versioned model storage with dependency tracking
- **SPCS Inference**: Containerized model serving on Snowpark Container Services
- **REST Gateway**: `ingress_enabled=True` exposes an HTTP endpoint for external clients

In [ ]:
import sys
sys.path.insert(0, "../source")
from snowpark_session import create_snowpark_session
from config import DATABASE, SCHEMA, WAREHOUSE, COMPUTE_POOL, MODEL_NAME, SERVICE_NAME

session = create_snowpark_session()
session.sql(f"USE WAREHOUSE {WAREHOUSE}").collect()
session.sql(f"USE DATABASE {DATABASE}").collect()
session.sql(f"USE SCHEMA {SCHEMA}").collect()

## Step 1: Train Model (quick retrain for this notebook's context)

In [ ]:
from training.train_model import load_training_data, train_model, FEATURE_COLUMNS, LABEL_COLUMN
import pandas as pd

df = load_training_data(session, DATABASE, SCHEMA)
model, metrics, importance = train_model(df)

# Prepare sample input for registry (schema inference)
available_features = [c for c in FEATURE_COLUMNS if c in df.columns]
sample_input = df[available_features].fillna(0).head(10)
print(f"\nSample input shape: {sample_input.shape}")

## Step 2: Register Model to Snowflake Model Registry

`reg.log_model()` stores the model, its dependencies, and input schema. The model becomes a first-class Snowflake object accessible via SQL.

In [ ]:
from registry.register_model import register_model

mv = register_model(
    model=model,
    sample_input=sample_input,
    version_name="V1",
    session=session,
    comment=f"XGBoost fraud detector - AUC={metrics['auc_roc']:.4f}, F1={metrics['f1']:.4f}",
)

# Show registered model info
from snowflake.ml.registry import Registry
reg = Registry(session=session, database_name=DATABASE, schema_name=SCHEMA)
print("\nRegistered models:")
reg.show_models()

## Step 3: Deploy Inference Service (SPCS with REST Gateway)

This creates a containerized inference endpoint on SPCS. With `ingress_enabled=True`, external clients can call the model via REST API.

In [ ]:
from serving.deploy_service import deploy_inference_service

mv = deploy_inference_service(version_name="V1", max_instances=2, session=session)

## Step 4: Test Inference

Once the service is ready (takes 2-5 minutes to build the container image), test with sample data.

In [ ]:
# Check service status (wait until READY)
import json
from serving.deploy_service import get_service_status

try:
    status = get_service_status(session=session)
    print(json.dumps(status, indent=2))
except Exception as e:
    print(f"Service still starting: {e}")
    print("Wait 2-5 minutes for image build, then re-run this cell.")

In [ ]:
# Test via Python SDK (once service is READY)
test_data = sample_input.head(5)
try:
    predictions = mv.run(test_data, function_name="predict_proba")
    print("Inference results:")
    print(predictions)
except Exception as e:
    print(f"Service not ready yet: {e}")

# Alternative: Test via SQL
print("\n--- SQL inference (works immediately via warehouse) ---")
session.sql(f"""
    SELECT {DATABASE}.{SCHEMA}.{MODEL_NAME}!PREDICT_PROBA(*)
    FROM (SELECT * FROM {DATABASE}.{SCHEMA}.RAW_TRANSACTIONS LIMIT 5)
""").show()